# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime, backup i migracja

Pracuj na projekcie `student/apps/lesson_g_app` w Android Studio.

Zaliczanie:
- `G01`: odpowiedz wysyła aplikacja automatycznie.
- `G02-G04`: wypełniasz formularz w notebooku i uruchamiasz komórkę wysyłki.


In [ ]:
# @title Dane studenta {"run":"auto","vertical-output":true,"display-mode":"form"}
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (auto-submit z aplikacji)

- Uprawnienie w manifeście nie oznacza automatycznie, że aplikacja dostanie dostęp.
- Dla wielu uprawnień Android wymaga osobnej zgody użytkownika w runtime.
- Dobre praktyki:
  1. prosić o uprawnienie dopiero gdy funkcja jest potrzebna,
  2. uzasadniać w UI, po co to jest,
  3. mieć fallback (aplikacja nie powinna się „wywracać” po odmowie).

## Jak  zrobić
1. Otwórz `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/AndroidManifest.xml`.
1. Przejrzyj wszystkie wpisy `<uses-permission ...>`.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Znajdź w UI sekcję „Student / Task 1” i zobacz, jakie uprawnienia aplikacja żąda po kliknięciu „Request permissions”.
1. Zrób mapę: każde uprawnienie -> jaka funkcja je wykorzystuje (mapa, zdjęcia, kamera, internet).
1. Sprawdź zachowanie fallback:
- co aplikacja pokazuje, gdy nie ma lokalizacji,
- co pokazuje, gdy brak dostępu do zdjęć,
- co pokazuje, gdy nie ma kamery.


### Gdy ID + wymagane uprawnienia są OK, aplikacja sama wyśle odpowiedź dla `G01`.


# G02 — APK / bundle provenance check

## Teoria
Każda aplikacja Android ma co najmniej dwa istotne identyfikatory:
- `applicationId` / `packageName`, czyli nazwę pakietu,
- tożsamość podpisu, czyli certyfikat użyty do podpisania APK lub AAB.

Dla bezpieczeństwa drugi z tych elementów jest ważniejszy. Dwie aplikacje mogą mieć podobną nazwę, podobny interfejs, a nawet zbliżony kod, ale jeżeli nie są podpisane oczekiwanym kluczem, to z punktu widzenia zaufania nie są „tą samą aplikacją”.

W praktyce atak repackagingu wygląda tak:
1. atakujący bierze oryginalne APK,
2. modyfikuje kod albo zasoby,
3. podpisuje zmieniony build własnym kluczem,
4. rozpowszechnia zmodyfikowaną wersję jako pozornie tę samą aplikację.

Z tego powodu w aplikacjach mobilnych rozróżnia się dwa etapy zaufania:
- **install-time trust**: Android sprawdza podpis przy instalacji i aktualizacji aplikacji,
- **runtime trust**: sama aplikacja albo backend wykonują dodatkową kontrolę i decydują, czy temu konkretnemu buildowi wolno zaufać.

## Zadanie
Dodaj do aplikacji mechanizm provenance check, który sprawdza tożsamość podpisu aktualnie uruchomionego builda i na tej podstawie wyznacza stan `ProvenanceState`.

Końcowy efekt ma być taki, że aplikacja:
- pobiera informację o podpisie z systemu,
- wyznacza stabilny identyfikator podpisu, np. fingerprint SHA-256,
- porównuje go z oczekiwaną wartością referencyjną,
- odrzuca build, który nie pasuje,
- rozdziela fakt „Android pozwolił aplikacji się uruchomić” od własnej decyzji runtime o zaufaniu do builda.

## Gdzie to zrobić
1. `app/src/main/java/com/example/secretlab/MainActivity.kt`
W tym pliku dodaj pomocniczą funkcję, na przykład w dolnej części pliku obok innych funkcji pomocniczych, która przyjmie `context: Context` i zwróci `ProvenanceState`.

2. `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`
W tym pliku zostaw model `ProvenanceState` i wykorzystaj go jako końcowy wynik sprawdzenia w `task2Check(...)`.

## Co zrobić
1. W `MainActivity.kt` dopisz funkcję pomocniczą, na przykład o kształcie: funkcja przyjmuje `context: Context` i zwraca `ProvenanceState`.
2. W tej funkcji pobierz `PackageManager` z kontekstu przez `context.packageManager`.
3. W tej samej funkcji pobierz nazwę własnego pakietu przez `context.packageName`.
4. Wywołaj `packageManager.getPackageInfo(context.packageName, PackageManager.GET_SIGNING_CERTIFICATES)`.
5. Z otrzymanego `PackageInfo` odczytaj `signingInfo`.
6. Z `signingInfo` pobierz listę podpisów przez `getApkContentsSigners()`.
7. Weź pierwszy podpis z tej listy jako aktywny podpis builda i pobierz jego bajty przez `toByteArray()`.
8. W tej samej funkcji policz z tych bajtów fingerprint SHA-256.
9. Zamień fingerprint na tekst w jednej, stabilnej postaci, na przykład ciąg znaków hex, żeby dało się go porównać z wartością referencyjną.
10. Obok tej logiki zdefiniuj stałą z oczekiwanym fingerprintem, z którą porównasz uruchomiony build.
11. Jeżeli fingerprint jest identyczny z oczekiwanym fingerprintem, ustaw `signingIdentityMatchesExpected = true`.
12. Jeżeli fingerprint nie pasuje, nie ma podpisu albo dane o podpisie są niespójne, ustaw `buildLooksTampered = true`.
13. Ustaw `installTimeTrustIsSeparatedFromRuntimeTrust = true` dopiero wtedy, gdy ta funkcja naprawdę wykona pełne sprawdzenie podpisu i zwróci wynik na podstawie odczytu z systemu.
14. Zwróć z tej funkcji kompletny `ProvenanceState`.
15. Następnie użyj tego stanu jako wejścia do `task2Check(...)`, żeby decyzja logiczna wynikała z rzeczywistego sprawdzenia podpisu, a nie z ręcznie wpisanych wartości.


## Weryfikacja
Po implementacji uruchom:
1. `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest` w Android Studio,
2. albo w terminalu, w katalogu `student/apps/lesson_g_app`:
   `./gradlew :app:bsmEvidence`

Jeżeli zadanie jest wykonane poprawnie, zobaczysz 5-znakowy kod dla `G02`.

## Dokumentacja
- `PackageManager`: https://developer.android.com/reference/android/content/pm/PackageManager
- `PackageManager.getPackageInfo(...)`: https://developer.android.com/reference/android/content/pm/PackageManager#getPackageInfo(java.lang.String,int)
- `PackageInfo.signingInfo`: https://developer.android.com/reference/android/content/pm/PackageInfo#signingInfo
- `SigningInfo`: https://developer.android.com/reference/android/content/pm/SigningInfo
- `SigningInfo.getApkContentsSigners()`: https://developer.android.com/reference/android/content/pm/SigningInfo#getApkContentsSigners()
- `Signature.toByteArray()`: https://developer.android.com/reference/android/content/pm/Signature#toByteArray()
- `MessageDigest`: https://developer.android.com/reference/java/security/MessageDigest

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
# @title G02 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true}
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)
zapisz_i_wyslij("G02", final_answer)

# G03 — Integrity-gated backend request

## Teoria
Sprawdzenie pochodzenia aplikacji nie wystarcza jeszcze do tego, żeby backend zaufał każdemu żądaniu wysłanemu przez klienta. W prawdziwych systemach backend zwykle oczekuje dwóch rzeczy jednocześnie:
- sygnału, że klient działa w zaufanym stanie,
- powiązania żądania z tożsamością aplikacji, tak aby nie dało się łatwo odtworzyć tego samego requestu z innego, niezaufanego klienta.

W tym zadaniu modelujesz właśnie taki przepływ. Nie implementujesz pełnego komercyjnego systemu typu Play Integrity, ale budujesz jego uproszczoną, testowalną wersję.

Request ma zostać dopuszczony tylko wtedy, gdy jednocześnie:
- `verdict` ma wartość pozwalającą na wykonanie operacji,
- tożsamość aplikacji zgadza się z tym, czego oczekuje logika bezpieczeństwa,
- request jest związany z tą tożsamością, a nie tylko wysłany na poprawny endpoint z poprawnym `taskId`.

## Zadanie
Dodaj do aplikacji bramkę bezpieczeństwa przed wykonaniem requestu backendowego. Ta bramka ma wyznaczać `IntegrityState`, a następnie blokować albo dopuszczać request.

Końcowy efekt ma być taki, że aplikacja:
- oblicza stan integralności / zaufania,
- podejmuje decyzję jeszcze przed wysłaniem danych,
- w razie braku zaufania nie wysyła nic i pokazuje bezpieczny fallback,
- nie traktuje sieciowej wysyłki jako czegoś, co „i tak warto spróbować”.

## Gdzie to zrobić
1. `app/src/main/java/com/example/secretlab/MainActivity.kt`
Tutaj wykonasz całą bramkę requestu. Szukaj funkcji `submitAnswer(...)`, bo to ona robi połączenie HTTP i to przed jej wywołaniem trzeba zatrzymać nieufny request.

2. `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`
Tu zostaw model `IntegrityState` i użyj `task3Check(...)` jako końcowej decyzji logicznej: wolno wysłać / nie wolno wysłać.

## Co dokładnie zrobić
1. W `MainActivity.kt` dopisz funkcję pomocniczą, która zbuduje `IntegrityState` dla odpowiedzi wysyłanej do backendu.
2. Ta funkcja powinna przyjąć co najmniej dane potrzebne do oceny requestu, na przykład `context`, `taskId`, `studentId` i wartość, którą chcesz wysłać.
3. Ustal, skąd bierze się `verdict`.
   W tym labie nie musi to być zewnętrzna usługa, ale ma to być realny sygnał logiczny typu „ALLOW” / „DENY”, wynikający z wykonanych sprawdzeń.
4. Ustal, jak wyliczasz `appPackageNameMatches`.
   Najprościej: odczytaj `context.packageName` i porównaj z oczekiwaną nazwą pakietu używaną przez aplikację.
5. Dodaj binding requestu do tożsamości aplikacji.
   To znaczy: zbuduj dodatkową wartość zależną od tożsamości klienta i danych requestu, zamiast wysyłać tylko `studentId`, `taskId` i treść odpowiedzi.
6. W praktyce zrób to tak, żeby przed wywołaniem `submitAnswer(...)` powstawał kompletny `IntegrityState`.
7. Następnie wywołaj `task3Check(...)` na tym stanie.
8. Jeżeli `task3Check(...)` zwróci `false`, przerwij ścieżkę przed `submitAnswer(...)`.
9. W tym przypadku ustaw też czytelny komunikat w UI, żeby użytkownik widział, że request został zablokowany przez brak zaufania, a nie przez przypadkowy błąd sieci.
10. Jeżeli `task3Check(...)` zwróci `true`, dopiero wtedy pozwól na wywołanie `submitAnswer(...)`.
11. Upewnij się, że auto-submit z `G01` nie korzysta przypadkiem z tej samej uproszczonej ścieżki i nie omija Twojej nowej logiki dla `G03`.


## Weryfikacja
Po implementacji uruchom:
1. `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest` w Android Studio,
2. albo w terminalu, w katalogu `student/apps/lesson_g_app`:
   `./gradlew :app:bsmEvidence`

Jeżeli zadanie jest wykonane poprawnie, zobaczysz 5-znakowy kod dla `G03`.

## Dokumentacja
- `HttpURLConnection`: https://developer.android.com/reference/java/net/HttpURLConnection
- `URL`: https://developer.android.com/reference/java/net/URL
- `URL.openConnection()`: https://developer.android.com/reference/java/net/URL#openConnection()

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
# @title G03 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)
zapisz_i_wyslij("G03", final_answer)

# G04 — Backup, migracja i higiena sekretów

## Teoria
W aplikacjach mobilnych sekrety mają cykl życia. Nie wystarczy wiedzieć, jak je odczytać w czasie działania programu. Trzeba jeszcze wiedzieć:
- skąd trafiają do aplikacji,
- gdzie są przechowywane,
- czy są zapisywane lokalnie,
- czy mogą zostać przeniesione na inne urządzenie,
- czy są przypadkiem ujawniane przez backup, migrację albo logowanie diagnostyczne.

To zadanie dotyczy właśnie tej warstwy. Masz uporządkować politykę obsługi sekretów w projekcie tak, żeby dane wrażliwe nie były przypadkowo traktowane jak zwykłe dane aplikacji.

Są tu dwa różne modele pracy z sekretami:
- model **build-time**: sekret trafia do aplikacji przez `local.properties` -> `build.gradle.kts` -> `BuildConfig` -> `AppSecrets` -> natywny helper w `secret_keys.cpp`,
- model **runtime/local storage**: sekret jest przechowywany lokalnie przez `AppSecretsStore` i `SecurePrefs`.

To rozróżnienie jest tutaj kluczowe. Build-time secret i runtime secret nie mają tych samych własności bezpieczeństwa. Build-time secret jest częścią procesu budowania aplikacji. Runtime secret może zostać zapisany na urządzeniu i wejść w interakcję z backupem, migracją, reinstalacją albo debugowaniem.

Dodatkowo trzeba pamiętać o jeszcze jednej rzeczy: to, że jakaś wartość jest ukryta za `BuildConfig`, zakodowana w blobie albo odszyfrowywana przez kod natywny, nie rozwiązuje automatycznie problemu backupu. Backup dotyczy przede wszystkim danych zapisanych przez aplikację na urządzeniu.

## Zadanie
Uporządkuj politykę sekretów i backupu tak, aby:
- rozróżniać sekrety dostarczane do buildu od sekretów zapisywanych lokalnie,
- wskazać, które dane nie powinny migrować między urządzeniami,
- poprawić konfigurację backupu tak, aby dane wrażliwe nie były objęte pustą, domyślną polityką,
- nie traktować „ukrycia sekretu w natywnym helperze” jako zastępstwa dla poprawnej polityki przechowywania i migracji.

## Gdzie to zrobić
1. `student/apps/lesson_g_app/local.properties`
Tu znajdują się wartości wejściowe przekazywane do buildu. Zwróć uwagę na `task4_secret_b64` oraz brak ustawionego `map_api_key_b64`.

2. `app/build.gradle.kts`
Znajdź `buildConfigField(...)` dla `MAP_API_KEY_B64` i `TASK4_SECRET_B64`. To jest miejsce, w którym sekret opuszcza warstwę pliku lokalnego i trafia do konfiguracji aplikacji.

3. `app/src/main/java/com/example/secretlab/secure/AppSecrets.kt`
Tu zobaczysz, że sekrety są odszyfrowywane przez `decryptBlob(...)`.

4. `app/src/main/cpp/secret_keys.cpp`
To tutaj jest rzeczywista logika odszyfrowania bloba. Przejdź funkcję `Java_com_example_secretlab_secure_AppSecrets_decryptBlob(...)` krok po kroku:
- wejściowy string jest najpierw dekodowany z Base64,
- wynik trafia do `xxteaDecrypt(...)`,
- dopiero końcowy plaintext wraca do warstwy Kotlin.

5. `app/src/main/java/com/example/secretlab/secure/AppSecretsStore.kt`
To jest drugi model: sekrety zapisane lokalnie po stronie aplikacji.

6. `app/src/main/AndroidManifest.xml`
Sprawdź `android:allowBackup="true"`.

7. `app/src/main/res/xml/backup_rules.xml`
Zobacz, że obecne reguły backupu są puste.

8. `app/src/main/java/com/example/secretlab/MainActivity.kt`
Znajdź `ApiMapCard(...)` i `buildStaticMapUrl(...)`. Zobaczysz tam, że aplikacja buduje URL do statycznej mapy Geoapify z parametrem `apiKey`, więc potrzebny jest klucz API Geoapify.

## Co dokładnie zrobić
1. Rozdziel analizę na dwa kanały:
- build-time secrets,
- runtime/local storage secrets.
2. Wskaż, które wartości należą do kanału build-time.
   W tym projekcie są to `MAP_API_KEY_B64` i `TASK4_SECRET_B64`, przechodzące przez `BuildConfig`, `AppSecrets` i natywny helper.
3. Wskaż, które wartości należą do kanału runtime.
   W tym projekcie odpowiada za to `AppSecretsStore` i warstwa `SecurePrefs`.
4. Określ, które dane mogą być bezpiecznie migrowane, a które nie powinny.
   Dla tego ćwiczenia klucz API mapy oraz sekret zadania 4 traktuj jako dane wrażliwe.
5. Pobierz darmowy klucz API Geoapify.
   Wejdź na stronę: https://myprojects.geoapify.com/ . Załóż konto albo zaloguj się, utwórz projekt i skopiuj klucz API przypisany do tego projektu. To jest plaintext klucza, którego później użyjesz do budowania URL-a statycznej mapy.
6. Przygotuj wartość dla `map_api_key_b64` dokładnie w formacie oczekiwanym przez aplikację.
   Nie wpisuj plaintextu do `local.properties`, bo `AppSecrets.readMapApiKey()` nie czyta surowego tekstu. Ta funkcja wywołuje `decryptBlob(...)`, więc w `map_api_key_b64` musi znaleźć się blob po szyfrowaniu XXTEA i po zakodowaniu do Base64.
7. Użyj gotowego skryptu z projektu, który przygotowuje taki blob.
   Otwórz terminal w katalogu `student/apps/lesson_g_app` i uruchom:
   `python3 tools/encrypt_secret_blob.py "TUTAJ_WKLEJ_SWÓJ_KLUCZ_GEOAPIFY"`
   Skrypt bierze plaintext, szyfruje go tym samym kluczem `kKey`, którego używa `secret_keys.cpp`, a potem wypisuje wynik w Base64. To właśnie ten wynik trzeba wkleić do `local.properties`.
8. Dodaj wygenerowaną wartość do `local.properties`.
   Dopisz linię:
   `map_api_key_b64=WYNIK_ZE_SKRYPTU`
   Nie dodawaj cudzysłowów. Po lewej stronie ma zostać dokładnie `map_api_key_b64`, po prawej ma znaleźć się cały string wypisany przez skrypt.
9. Przebuduj aplikację i uruchom ją ponownie.
10. Sprawdź efekt końcowy: karta mapy przestaje wyświetlać komunikat `Map API key missing.` i zaczyna ładować statyczną mapę z Geoapify.
11. Oceń, czy obecna konfiguracja backupu jest akceptowalna.
   Przy `android:allowBackup="true"` i pustym `backup_rules.xml` odpowiedź powinna być krytyczna, bo aplikacja nie rozróżnia jeszcze, które dane lokalne wolno kopiować, a których nie.
12. Popraw politykę backupu.
   Masz dwie sensowne drogi:
- wyłączyć backup całkowicie,
- zostawić backup, ale jawnie wykluczyć lokalne dane wrażliwe.
13. Upewnij się, że nie mieszasz build-time secret z runtime secret.
   To, że blob jest ukryty w kodzie natywnym, nie rozwiązuje problemu danych zapisanych lokalnie po stronie aplikacji.
14. Przejrzyj też miejsca, w których sekret mógłby wyciec przez diagnostykę, logi albo debug output. Polityka higieny sekretów nie kończy się na samym backupie.

## Dodatkowa lektura pomocnicza
- https://al-e-shevelev.medium.com/a-secure-way-to-store-api-keys-in-android-applications-238135709067
- https://apidocs.geoapify.com/docs/maps/static/

Te materiały traktuj pomocniczo. Nie chodzi o mechaniczne powtórzenie wzorca „wrzuć sekret do C++ i problem znika”. W tym zadaniu chodzi o całą politykę: skąd sekret trafia do builda, co dzieje się z nim w runtime i czy może zostać przeniesiony na inne urządzenie.

## Weryfikacja
Po wykonaniu zadania końcowy projekt powinien spełniać jednocześnie wszystkie warunki:
- aplikacja potrafi odszyfrować `TASK4_SECRET_B64` i odczytać 5-znakowy sekret Task 4,
- aplikacja potrafi odszyfrować `MAP_API_KEY_B64` i po dodaniu poprawnego bloba ładuje mapę zamiast komunikatu o braku klucza,
- polityka backupu nie pozostawia lokalnych danych wrażliwych pod pustą, domyślną konfiguracją,
- rozróżnienie między build-time secret i runtime secret jest zachowane.

Żeby odczytać odpowiedź do notebooka, przejdź ścieżkę sekretu Task 4:
`local.properties` -> `build.gradle.kts` -> `BuildConfig.TASK4_SECRET_B64` -> `AppSecrets.readTask4Secret()` -> `decryptBlob(...)` w `secret_keys.cpp`.

Na końcu odczytaj odszyfrowaną, 5-znakową wartość sekretu Task 4. To właśnie tę wartość wpisujesz do formularza odpowiedzi.

## Dokumentacja
- `android:allowBackup`: https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`: https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `BuildConfig`: https://developer.android.com/build/gradle-tips#share-custom-fields-and-resource-values-with-your-apps-code
- `System.loadLibrary(...)`: https://developer.android.com/reference/java/lang/System#loadLibrary(java.lang.String)

## Format odpowiedzi
Do `final_answer` wpisujesz tylko 5-znakową wartość sekretu dla `G04`.


In [ ]:
# @title G04 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)
zapisz_i_wyslij("G04", final_answer)